In [55]:
import kagglehub
import shutil
import os
import pandas as pd
import csv
import numpy as np

# Download to cache
#cache_path = kagglehub.dataset_download("sharmajicoder/gaming-and-mental-health")
cache_path = kagglehub.dataset_download("imadkhattak/airbnb-data")

# Define your repo data folder
repo_data_path = "./data"

# Create folder if it doesn't exist
os.makedirs(repo_data_path, exist_ok=True)

# Copy files from cache to repo
for file_name in os.listdir(cache_path):
    shutil.copy(os.path.join(cache_path, file_name), repo_data_path)

print("Dataset copied to:", repo_data_path)

Dataset copied to: ./data


In [56]:
import pandas as pd
# Load the CSV file into a DataFrame
df = pd.read_csv("./data/train_regression.csv")
# Display the first few rows of the DataFrame
df.head(10)

,id,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,5.387292e+07,366516391,4/9/2018,"Chicago, IL",within an hour,96%,95%,f,Back of the Yards,11,...,5.00,5.00,4.28,4.83,f,9,1,8,0,0.33
1,3.926966e+07,50276775,12/23/2014,NaN,within an hour,100%,97%,f,Clearwater Beach,1141,...,NaN,NaN,NaN,NaN,t,58,0,58,0,NaN
2,5.564810e+17,681680651,3/8/2020,NaN,within an hour,100%,100%,f,Lake View East,1,...,5.00,5.00,4.93,4.80,f,1,1,0,0,2.78
3,5.239728e+07,94345291,10/22/2015,"Chicago, IL",within an hour,100%,98%,f,Bucktown,55,...,4.38,4.23,4.54,3.85,t,55,55,0,0,3.61
4,7.459490e+17,790998106,4/4/2021,"Chicago, IL",within an hour,100%,100%,t,Near South Side,74,...,4.83,4.85,4.74,4.72,t,74,74,0,0,3.24
5,7.407770e+17,50276775,12/23/2014,NaN,within an hour,100%,97%,f,Clearwater Beach,1141,...,5.00,5.00,4.00,5.00,t,58,0,58,0,1.18
6,8.416530e+17,570167059,8/12/2019,"Chicago, IL",within a few hours,100%,97%,t,Hyde Park,3,...,4.69,4.86,4.43,4.63,f,3,3,0,0,0.88
7,7.645760e+17,975964114,11/16/2022,NaN,within an hour,100%,99%,t,Humboldt Park,4,...,4.84,4.92,4.40,4.72,f,4,4,0,0,2.43
8,8.564280e+17,44110882,10/2/2014,"Chicago, IL",within a few hours,100%,93%,f,Douglas,1,...,4.88,4.94,4.78,4.91,f,1,1,0,0,1.92
9,8.624450e+17,497520971,3/14/2019,NaN,within an hour,99%,99%,t,West Loop/Greektown,71,...,4.85,5.00,5.00,4.69,t,51,51,0,0,1.38


In [57]:
train = pd.read_csv("./data/train_regression.csv")
test = pd.read_csv("./data/test_regression.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


Train shape: (5000, 54)
Test shape: (3338, 53)


In [58]:
missing_df = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": df.isna().mean() * 100
}).sort_values("Missing_Percent", ascending=False)

missing_df

,Missing_Count,Missing_Percent
reviews_per_month,1042,20.84
review_scores_cleanliness,1042,20.84
first_review,1042,20.84
last_review,1042,20.84
review_scores_communication,1042,20.84
review_scores_accuracy,1041,20.82
review_scores_value,1041,20.82
review_scores_rating,1041,20.82
review_scores_checkin,1041,20.82
review_scores_location,1041,20.82


# GUIDE SETUP

In [59]:
original_cols_to_keep = [
    'host_response_rate',
    'host_acceptance_rate',
    'host_is_superhost',
    'accommodates',
    'bathrooms_text',
    'beds',
    'price',
    'minimum_nights',
    'maximum_nights',
    'has_availability',
    'number_of_reviews',
    'review_scores_rating',
    'instant_bookable',
    'reviews_per_month'
]

clean_df = df[original_cols_to_keep].copy()

# modify bathrooms_text to extract numeric value
clean_df["bathrooms"] = (
    clean_df["bathrooms_text"]
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# PRICE: Remove $, commas, spaces
clean_df["price"] = (
    clean_df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(float)
)

# RATES: Remove % and convert to float
clean_df["host_acceptance_rate"] = (
    clean_df["host_acceptance_rate"]
    .str.replace("%", "", regex=False)
    .astype(float)
    / 100
)
clean_df["host_response_rate"] = (
    clean_df["host_response_rate"]
    .str.replace("%", "", regex=False)
    .astype(float)
    / 100
)

clean_df["price"] = np.log(clean_df["price"])

# drop certain columns
cols_to_remove = [
    "bathrooms_text"
]

clean_df = clean_df.drop(columns=cols_to_remove)

clean_df.head()


,host_response_rate,host_acceptance_rate,host_is_superhost,accommodates,beds,price,minimum_nights,maximum_nights,has_availability,number_of_reviews,review_scores_rating,instant_bookable,reviews_per_month,bathrooms
0,0.96,0.95,f,1,1.0,3.401197,32,1125,t,18,4.94,f,0.33,1.0
1,1.00,0.97,f,12,3.0,7.128496,32,365,t,0,NaN,t,NaN,3.0
2,1.00,1.00,f,6,3.0,5.365976,2,45,t,14,4.87,f,2.78,1.0
3,1.00,0.98,f,2,1.0,4.077537,2,180,t,13,4.08,t,3.61,1.0
4,1.00,1.00,t,6,3.0,5.017280,2,365,t,64,4.80,t,3.24,2.0


In [60]:
# airbnb.dsc

# SETTINGS
dsc_filename = "airbnb.dsc"
data_filename = "airbnb.txt"
missing_value = "NA"

# Class
class_count = 2

# WRITE DATA FILE
setup_df = clean_df.fillna(missing_value)

setup_df.to_csv(
    data_filename,
    sep=" ",
    index=False,
    header=True
)

# DETERMINE VARIABLE TYPES
lines = []
target_column = "price"

for i, col in enumerate(clean_df.columns, start=1):
    if col == target_column:
        # Dependent variable or death indicator variable. TARGET
        var_type = "d"
    elif pd.api.types.is_numeric_dtype(clean_df[col]):
        # Numerical variable used both for splitting the nodes and for fitting 
        # the node regression models
        var_type = "n"
    else:
        # Categorical variable used both for splitting and for node modeling in regression
        var_type = "b"
    lines.append(f"{i} {col} {var_type}")

# WRITE DSC FILE
with open(dsc_filename, "w") as f:
    f.write(f"{data_filename}\n")
    f.write(f"{missing_value}\n")
    f.write(f"{class_count}\n")
    for line in lines:
        f.write(line + "\n")

print("DSC file created successfully.")


DSC file created successfully.


In [61]:
len(setup_df.columns.tolist())

14

In [62]:
setup_df.columns.tolist()

['host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'accommodates',
 'beds',
 'price',
 'minimum_nights',
 'maximum_nights',
 'has_availability',
 'number_of_reviews',
 'review_scores_rating',
 'instant_bookable',
 'reviews_per_month',
 'bathrooms']